# TP3 — Classifieurs et méthodes d'ensemble
**CNFSDP — Introduction au Machine Learning** · L3 S5 · 2026-2027 · Rois Céti DIMBAMBA · chapitre 3 · 45 min en salle

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rdimbamba/ml-cnfsdp/blob/main/notebooks/TP3.ipynb)

Partie A : plus proches voisins et régression logistique. Partie B : arbres et indice de Gini. Partie C : bagging, forêt, AdaBoost.
Données : le fil rouge du cours (10 enfants, chiffres fictifs), des « croissants » simulés et `breast_cancer`, fourni avec scikit-learn (aucun téléchargement).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier
VERT, OR, GRIS = "#126B2A", "#B8860B", "#8A8A8A"   # couleurs du cours

# Fil rouge : études de la mère (ans), distance au centre de santé (km), vacciné
X0 = np.array([[0, 9], [1, 3], [3, 7], [4, 2], [6, 8], [6, 4], [7, 9], [9, 6], [10, 1], [2, 5]])
y0 = np.array([0, 1, 0, 1, 0, 1, 1, 1, 1, 0])

## Partie A — Plus proches voisins et régression logistique
### Question 1 — $k$-NN à la main
Pour le nouvel enfant $x=(5,5)$ : calculer les distances au carré aux 10 enfants, trier, puis donner $\hat\eta(x)$ et la prédiction pour $k=1,3,5$. Vérifier avec `KNeighborsClassifier`.

In [ ]:
x = np.array([5, 5])
# À compléter : distances au carré, tri (np.argsort), proportion de 1 parmi les k premiers,
# puis comparaison avec KNeighborsClassifier(n_neighbors=k).fit(X0, y0).predict_proba([x])


### Question 2 — Choisir $k$ par validation croisée
Sur 300 points en croissants (`make_moons(noise=0.3)`), calculer l'erreur de validation croisée à 5 blocs pour $k=1,\dots,50$, tracer la courbe et donner le meilleur $k$. Mesurer ensuite l'erreur test sur 5 000 nouveaux points.

In [ ]:
X, y = make_moons(n_samples=300, noise=0.3, random_state=0)
Xt, yt = make_moons(n_samples=5000, noise=0.3, random_state=1)
# À compléter : erreur de CV (1 - cross_val_score(...).mean()) pour k = 1..50, tracé, k optimal, erreur test


### Question 3 — Régression logistique sur le fil rouge
Avec la seule variable « études », ajuster une régression logistique sans pénalité (`LogisticRegression(C=1e8)`). Retrouver $\hat\theta\approx(-1{,}46;\,0{,}43)$, la frontière $-\hat\theta_0/\hat\theta_1$, la cote $e^{\hat\theta_1}$ et la perte logistique moyenne. Vérifier que le gradient $\frac1n\sum_i(\hat\eta_i-y_i)x_{ij}$ est (presque) nul.

In [ ]:
xe = X0[:, [0]]                                  # années d'études seules
# À compléter : LogisticRegression(C=1e8), intercept_ et coef_, frontière, cote, perte logistique, gradient


## Partie B — Arbres et indice de Gini
### Question 4 — La meilleure coupure
Écrire une fonction `gini(y)` qui renvoie $2p(1-p)$, puis une fonction qui calcule l'impureté moyenne d'une coupure « $x_j\le s$ ». Essayer tous les seuils (milieux entre valeurs consécutives) des deux variables du fil rouge et retrouver la coupure du cours (distance $\le4{,}5$, impureté $0{,}267$). Comparer avec l'arbre de profondeur 2 de scikit-learn.

In [ ]:
def gini(y):
    # À compléter
    pass

def impurete(X, y, j, s):
    # À compléter : impureté moyenne de la coupure X[:, j] <= s
    pass

# À compléter : boucle sur les deux variables et les seuils, tri, puis arbre de profondeur 2 (export_text)


## Partie C — Méthodes d'ensemble sur `breast_cancer`
569 tumeurs, 30 mesures ; $y=1$ si la tumeur est bénigne. Découpage 70 % / 30 %, stratifié.

In [ ]:
D = load_breast_cancer()
Xa, Xt, ya, yt = train_test_split(D.data, D.target, test_size=0.3, random_state=0, stratify=D.target)
print(Xa.shape, Xt.shape)

### Question 5 — Le bagging écrit en NumPy
Pour $B=100$ : tirer un échantillon bootstrap (indices avec remise), y apprendre un arbre profond, prédire le test ; agréger par vote majoritaire. Comparer l'exactitude test avec un arbre seul et avec `BaggingClassifier`. Tracer l'exactitude du vote en fonction du nombre d'arbres.

In [ ]:
rng = np.random.default_rng(0)
B, n = 100, len(ya)
votes = np.zeros(len(yt))
exactitude = []
# À compléter : boucle bootstrap (rng.integers(0, n, n)), arbre profond, cumul des votes,
# exactitude du vote majoritaire ; comparaison avec un arbre seul et BaggingClassifier


### Question 6 — Forêt aléatoire : out-of-bag et importances
Ajuster `RandomForestClassifier(n_estimators=300, oob_score=True)`. Comparer l'exactitude out-of-bag à l'exactitude test, puis afficher les cinq variables les plus importantes.

In [ ]:
# À compléter : RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0),
# oob_score_ contre score test, cinq plus grandes feature_importances_


### Question 7 — AdaBoost à la main
Six points $x=1,\dots,6$, classes $y=(+1,+1,-1,+1,+1,-1)$. Programmer trois tours d'AdaBoost avec des arbres à une coupure (`DecisionTreeClassifier(max_depth=1)` accepte des poids : `fit(x, y, sample_weight=w)`). À chaque tour, afficher la coupure, $\varepsilon_m$, $\alpha_m$ et les nouveaux poids ; vérifier que les mal classés pèsent $\frac12$ après la mise à jour. Comparer avec `AdaBoostClassifier`, dont les poids valent $2\alpha_m$.

In [ ]:
x6 = np.arange(1, 7).reshape(-1, 1); y6 = np.array([1, 1, -1, 1, 1, -1])
w = np.full(6, 1 / 6); F = np.zeros(6)
# À compléter : trois tours (arbre à une coupure pondéré, eps, alpha, mise à jour et renormalisation des poids, F)
# puis comparaison avec AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=3)


**À commenter :** que gagne-t-on à combiner des arbres ? Pourquoi les poids des points mal classés augmentent-ils ?